# Drone env — browser eval

Pick a checkpoint, click **Build & load**. This copies the chosen weights to `resources/drone/drone_weights.bin` and runs `bash build.sh drone --web`, which bakes them into a WASM bundle. The iframe below reloads against it.

In [ ]:
import subprocess, atexit, time, pathlib, shutil, ipywidgets as W
from IPython.display import IFrame, display, clear_output

def find_project_root(start: pathlib.Path = None) -> pathlib.Path:
    start = start or pathlib.Path.cwd()
    for p in [start, *start.parents]:
        if (p / 'build.sh').exists():
            return p
    raise RuntimeError(f'No build.sh found above {start}')

PROJECT = find_project_root()
BUILD   = PROJECT / 'build' / 'web' / 'drone'   # build.sh drone --web writes game.html here
WEIGHTS = PROJECT / 'resources' / 'drone' / 'drone_weights.bin'  # baked into the web build
PORT    = 8765

BUILD.mkdir(parents=True, exist_ok=True)
srv = subprocess.Popen(
    ['python3', '-m', 'http.server', '-d', str(BUILD), str(PORT)],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
atexit.register(srv.terminate)
time.sleep(0.3)

def list_checkpoints():
    base = PROJECT / 'checkpoints/drone'
    return sorted(base.rglob('*.bin'), key=lambda p: p.stat().st_mtime, reverse=True)

def _emsdk_env():
    # build.sh --web needs emcc on PATH; a running Jupyter kernel won't have it even
    # after you source emsdk_env.sh in a terminal, so source it into the build
    # subprocess instead (this also puts the node emcc needs on PATH).
    candidates = ['/workspace/emsdk/emsdk_env.sh',
                  str(pathlib.Path.home() / 'emsdk' / 'emsdk_env.sh')]
    return next((c for c in candidates if pathlib.Path(c).exists()), None)

def rebuild(model_path):
    # build.sh bakes resources/drone/drone_weights.bin into the WASM bundle, so
    # stage the chosen checkpoint there first, then compile for web.
    shutil.copy(model_path, WEIGHTS)
    env = _emsdk_env()
    prefix = f'source "{env}" >/dev/null 2>&1 && ' if env else ''
    return subprocess.run(
        ['bash', '-c', f'{prefix}bash build.sh drone --web'],
        cwd=PROJECT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )

print(f'Project root: {PROJECT}')
print(f'Server up on :{PORT}')
print(f'emsdk_env.sh: {_emsdk_env() or "NOT FOUND — install emsdk first"}')

In [ ]:
ckpts = list_checkpoints()
if not ckpts:
    print('No checkpoints in checkpoints/drone/. Train one first: puffer train drone')
else:
    def label(p, is_latest):
        run = p.parent.name
        try:
            step = f'{int(p.stem):,} steps'
        except ValueError:
            step = p.stem
        tag = '  ← latest' if is_latest else ''
        return f'{run} · {step}{tag}'

    options = [(label(p, i == 0), str(p)) for i, p in enumerate(ckpts)]
    picker = W.Dropdown(options=options, value=str(ckpts[0]),
                        description='Model:', layout=W.Layout(width='80%'))
    button = W.Button(description='Build & load', button_style='primary')
    status = W.Output()
    frame  = W.Output()

    def on_click(_):
        with status:
            clear_output(); print(f'Building with {picker.value} ...')
        res = rebuild(picker.value)
        with status:
            clear_output()
            if res.returncode != 0:
                print('Build failed:\n' + res.stdout); return
            print(f'Loaded: {picker.value}')
        with frame:
            clear_output()
            display(IFrame(f'/proxy/{PORT}/game.html?v={int(time.time())}', width=960, height=640))

    button.on_click(on_click)
    display(W.VBox([W.HBox([picker, button]), status, frame]))
    on_click(None)